<center> <h1><b>Analysis and Machine Learning-Based Detection of Brute-Force Attacks in IIoT</b></h1> </center>

**Author(s):** Yassin BIBRINE & Ali ACHENAN   
**Abstract:** This notebook presents a comprehensive analysis of brute-force attacks within Industrial Internet of Things (IIoT) networks. By processing network traffic captures using advanced tools like Zeek, we structure raw data into actionable datasets for machine learning-based intrusion detection. The workflow emphasizes methodological rigor, reproducibility, and the integration of protocol-aware features to distinguish malicious authentication attempts from benign traffic. Two dataset versions (V1: comprehensive merge; V2: targeted brute-force logs) are generated to support exploratory analysis and model training.

## 1. Introduction

### 1.1 Background and Motivation

The **Industrial Internet of Things (IIoT)** integrates sensors, actuators, and industrial control systems such as *Programmable Logic Controllers (PLCs)* and *SCADA systems* into interconnected networked environments. This convergence enables automation, real-time monitoring, and data-driven decision-making in critical sectors like manufacturing and energy. However, it also exposes legacy systems and resource-constrained devices to cybersecurity threats, including **Distributed Denial-of-Service (DDoS) attacks**, reconnaissance scanning, web-based injections, and unauthorized access attempts.

The motivation for this research stems from the increasing vulnerability of IIoT ecosystems. Traditional security measures often fail against adaptive threats, necessitating data-driven approaches. By analyzing real-world IIoT network traffic encompassing both benign and malicious behaviors we aim to uncover attack dynamics, identify behavioral patterns, and develop **Intrusion Detection Systems (IDS)** powered by machine learning for enhanced adaptability and accuracy.

### 1.2 Research Focus: Brute-Force Attacks

Among IIoT threats, **brute-force attacks** are prevalent, involving systematic trial-and-error of username/password combinations to gain unauthorized access. These attacks target services like **Secure Shell (SSH)** for remote access, **HTTP** for web interfaces, and **MQTT** for authentication-dependent messaging in IIoT. Successful breaches can result in process manipulation, data exfiltration, or lateral movement, potentially disrupting critical infrastructure.

Understanding brute-force dynamics requires examining high-rate authentication failures, session durations, and traffic anomalies concepts rooted in behavioral anomaly detection theory.

### 1.3 Analytical and Machine Learning Objectives

The primary objective is to design and train machine learning models for predicting brute-force attacks in IIoT networks. This involves:
1. Structuring raw network traffic into protocol-aware event logs.
2. Analyzing authentication behaviors and traffic patterns.
3. Developing predictive models to classify malicious vs. legitimate access.

Emphasis is placed on reproducibility, robustness, and cross-validation to ensure models generalize across diverse IIoT deployments.

## 2. Methodology

### 2.1 Tools and Technologies

#### Zeek Network Security Monitor
- A passive analysis framework that transforms raw packet captures (PCAPs) into semantic, protocol-specific logs (`conn.log` for connections, `ssh.log` for authentication events).
- Outputs in line-delimited JSON for seamless integration into data pipelines.
- Chosen over alternatives like TShark due to its behavioral abstraction: Zeek extracts security-relevant features such as authentication failures and session metadata at a higher level, simplifying brute-force pattern detection without manual packet reconstruction.

#### Python and Pandas
- Employed for ingesting JSON logs, preserving protocol fields, and enriching with metadata (device type, PCAP source).
- Facilitates merging into structured CSVs, enabling exploratory data analysis (EDA) and machine learning preprocessing.

### 2.2 Data Pipeline Overview

The pipeline follows: Raw PCAPs → **Zeek** (JSON log generation) → **Python/Pandas** (structuring and annotation) → **Analysis and ML Modeling**.

This approach ensures traceability and scalability, addressing challenges like data volume and protocol diversity in IIoT traffic.

## 3. Data Preparation

### 3.1 Phase 1: Comprehensive Data Merging (V1)

In this phase, all Zeek logs from multiple PCAP folders are merged into a single CSV (`merged_data_V1.csv`). This version provides a holistic view of network events, including metadata for contextual analysis. It is ideal for broad exploratory studies but may include irrelevant logs for targeted brute-force detection.

#### 3.1.1 Imports and Setup

We import essential libraries and define paths. `glob` is used for pattern-matching log files, ensuring efficient traversal of directory structures.

In [ ]:
import os
import pandas as pd
from glob import glob

# Path to your 8 folders
base_path = "zeek_logs"

# Where the final CSV will be saved
output_csv = "merged_data_V1.csv"

#### 3.1.2 Folder Discovery

Identify PCAP folders, each representing a device or attack scenario. This modular approach supports scalability for larger datasets.

In [ ]:
# Each folder corresponds to one device / PCAP
pcap_folders = [f.path for f in os.scandir(base_path) if f.is_dir()]
pcap_folders

['zeek_logs\\attack_bruteforce_dictionary-ssh_ap',
 'zeek_logs\\attack_bruteforce_dictionary-ssh_edge1',
 'zeek_logs\\attack_bruteforce_dictionary-ssh_mqtt-broker',
 'zeek_logs\\attack_bruteforce_dictionary-ssh_router',
 'zeek_logs\\attack_bruteforce_dictionary-ssh_switch',
 'zeek_logs\\attack_bruteforce_dictionary-telnet_ap',
 'zeek_logs\\attack_bruteforce_dictionary-telnet_edge1',
 'zeek_logs\\attack_bruteforce_dictionary-telnet_mqtt-broker']

#### 3.1.3 Helper Function for Log Reading

This function reads Zeek's JSON logs, handling line-delimited formats common in network analysis. Metadata enrichment (`pcap_id`, `device_type`) facilitates downstream filtering and grouping, crucial for multi-device IIoT studies.

In [ ]:
def read_log_file_json(file_path, log_type, pcap_id, device_type):
    """
    Reads a Zeek JSON log file (.log) and adds metadata columns
    """
    try:
        # Read JSON lines (Zeek JSON logs are usually line-delimited JSON)
        df = pd.read_json(file_path, lines=True)
    except Exception as e:
        print(f"Failed to read {file_path}: {e}")
        return pd.DataFrame()

    # Add metadata
    df['pcap_id'] = pcap_id
    df['device_type'] = device_type
    df['log_type'] = log_type

    return df

#### 3.1.4 Example Processing for a Single Folder

Demonstrate log concatenation for one folder. This step verifies data integrity, revealing fields like timestamps (`ts`), connection UIDs, and protocol details—key for temporal analysis in attack detection.

In [ ]:
example_folder = pcap_folders[0]
device_type = os.path.basename(example_folder)
all_logs = []

for log_file in glob(os.path.join(example_folder, "*.log")):
    log_type = os.path.splitext(os.path.basename(log_file))[0]
    df_log = read_log_file_json(log_file, log_type, pcap_id=device_type, device_type=device_type)
    if not df_log.empty:
        all_logs.append(df_log)

folder_df = pd.concat(all_logs, ignore_index=True)
folder_df.head()

,ts,analyzer_kind,analyzer_name,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,failure_reason,...,host_key_alg,host_key,auth_success,cipher,resumed,established,ssl_history,name,notice,peer
0,1.739537e+09,packet,TEREDO,CwwNM63C3opLA8MFR2,192.168.1.205,38822.0,255.255.255.255,29810.0,udp,Truncated Teredo or invalid inner IP version,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.739537e+09,NaN,NaN,CGxBxl3en6yMbdxKA8,192.168.1.87,50336.0,54.69.37.156,443.0,tcp,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.739537e+09,NaN,NaN,CwzJvg1V59ReXXjaX4,192.168.1.100,39252.0,192.168.1.205,22.0,tcp,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.739537e+09,NaN,NaN,CfFBkf1cfOee8KQDra,192.168.1.102,58918.0,8.8.8.8,53.0,udp,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.739537e+09,NaN,NaN,C9LdHVV0aUXR0rvT9,192.168.1.52,43544.0,192.168.1.1,1900.0,tcp,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### 3.1.5 Descriptive Statistics for Example Folder

Generate summary statistics to inspect data distribution, identifying outliers in features like duration or byte counts potential indicators of brute-force persistence.

In [ ]:
folder_df.describe()

,ts,id.orig_p,id.resp_p,duration,orig_bytes,resp_bytes,missed_bytes,orig_pkts,orig_ip_bytes,resp_pkts,...,total_bytes,missing_bytes,overflow_bytes,trans_depth,request_body_len,response_body_len,status_code,payload_len,auth_attempts,auth_success
count,2.413000e+03,2412.000000,2412.000000,295.000000,293.000000,293.000000,325.000000,325.000000,325.000000,325.000000,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1838.000000,75.000000,4.0
mean,1.739548e+09,52336.210199,1886.762438,21.303813,15706.634812,37239.204778,0.590769,141.427692,20964.175385,139.870769,...,134.0,0.0,0.0,1.0,0.0,134.0,404.0,4.844396,0.346667,0.0
std,5.348213e+05,7873.278583,3254.432565,40.669371,32273.205074,75872.531500,7.519229,285.587588,42829.427294,286.627606,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.234774,1.474956,0.0
min,1.739537e+09,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,134.0,0.0,0.0,1.0,0.0,134.0,404.0,0.000000,0.000000,0.0
25%,1.739537e+09,50692.000000,1883.000000,0.023856,44.000000,44.000000,0.000000,1.000000,72.000000,1.000000,...,134.0,0.0,0.0,1.0,0.0,134.0,404.0,2.000000,0.000000,0.0
50%,1.739537e+09,52268.000000,1883.000000,0.025027,44.000000,44.000000,0.000000,1.000000,72.000000,1.000000,...,134.0,0.0,0.0,1.0,0.0,134.0,404.0,3.000000,0.000000,0.0
75%,1.739537e+09,55866.000000,1883.000000,7.803232,4512.000000,1018.000000,0.000000,24.000000,4800.000000,11.000000,...,134.0,0.0,0.0,1.0,0.0,134.0,404.0,5.000000,0.000000,0.0
max,1.765808e+09,64829.000000,53611.000000,121.569801,270938.000000,194730.000000,96.000000,772.000000,137724.000000,780.000000,...,134.0,0.0,0.0,1.0,0.0,134.0,404.0,16.000000,7.000000,0.0


#### 3.1.6 Full Dataset Merging (V1)

Aggregate logs across all folders into a master DataFrame. This results in a wide dataset (many columns due to varying log schemas), suitable for feature engineering in ML pipelines.

In [ ]:
all_folders_dfs = []

for folder in pcap_folders:
    device_type = os.path.basename(folder)
    folder_logs = []

    for log_file in glob(os.path.join(folder, "*.log")):
        log_type = os.path.splitext(os.path.basename(log_file))[0]
        df_log = read_log_file_json(log_file, log_type, pcap_id=device_type, device_type=device_type)
        if not df_log.empty:
            folder_logs.append(df_log)

    if folder_logs:
        folder_df = pd.concat(folder_logs, ignore_index=True)
        all_folders_dfs.append(folder_df)

master_df = pd.concat(all_folders_dfs, ignore_index=True)
master_df.head()

,ts,analyzer_kind,analyzer_name,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,failure_reason,...,certificate.key_alg,certificate.sig_alg,certificate.key_type,certificate.key_length,certificate.exponent,host_cert,client_cert,san.dns,basic_constraints.ca,basic_constraints.path_len
0,1.739537e+09,packet,TEREDO,CwwNM63C3opLA8MFR2,192.168.1.205,38822.0,255.255.255.255,29810.0,udp,Truncated Teredo or invalid inner IP version,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.739537e+09,NaN,NaN,CGxBxl3en6yMbdxKA8,192.168.1.87,50336.0,54.69.37.156,443.0,tcp,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.739537e+09,NaN,NaN,CwzJvg1V59ReXXjaX4,192.168.1.100,39252.0,192.168.1.205,22.0,tcp,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.739537e+09,NaN,NaN,CfFBkf1cfOee8KQDra,192.168.1.102,58918.0,8.8.8.8,53.0,udp,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.739537e+09,NaN,NaN,C9LdHVV0aUXR0rvT9,192.168.1.52,43544.0,192.168.1.1,1900.0,tcp,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### 3.1.7 Export V1 Dataset

Save the merged dataset for persistence. Note: JSON logs were preferred over raw metadata for their rich behavioral insights, aligning with brute-force detection needs.

In [ ]:
master_df.to_csv(output_csv, index=False)
print(f"Merged CSV saved to {output_csv}")

Merged CSV saved to merged_data_V1.csv


#### 3.1.8 Dataset Dimensions

Inspect shape to confirm successful merging, highlighting the dataset's scale for ML feasibility.

In [ ]:
master_df.shape

(35988, 130)

#### 3.1.9 Preview Merged Data

Display header to verify the structure, noting that sparse columns of record variation are a common challenge in network data fusion.

In [ ]:
master_df.head()

,ts,analyzer_kind,analyzer_name,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,failure_reason,...,certificate.key_alg,certificate.sig_alg,certificate.key_type,certificate.key_length,certificate.exponent,host_cert,client_cert,san.dns,basic_constraints.ca,basic_constraints.path_len
0,1.739537e+09,packet,TEREDO,CwwNM63C3opLA8MFR2,192.168.1.205,38822.0,255.255.255.255,29810.0,udp,Truncated Teredo or invalid inner IP version,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.739537e+09,NaN,NaN,CGxBxl3en6yMbdxKA8,192.168.1.87,50336.0,54.69.37.156,443.0,tcp,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.739537e+09,NaN,NaN,CwzJvg1V59ReXXjaX4,192.168.1.100,39252.0,192.168.1.205,22.0,tcp,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.739537e+09,NaN,NaN,CfFBkf1cfOee8KQDra,192.168.1.102,58918.0,8.8.8.8,53.0,udp,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.739537e+09,NaN,NaN,C9LdHVV0aUXR0rvT9,192.168.1.52,43544.0,192.168.1.1,1900.0,tcp,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 3.2 Optimized Data Preparation (V2: Brute-Force Relevant Logs)

V2 filters for logs directly pertinent to brute-force detection such as SSH, HTTP, MQTT and connections. This reduces noise, focusing on features like authentication attempts and payload lengths essential for time-window-based anomaly detection models.

#### 3.2.1 Define Relevant Log Types

Select logs capturing authentication events, informed by protocol vulnerabilities in IIoT.

In [ ]:
BRUTEFORCE_LOG_TYPES = {
    "ssh",
    "conn",
    "http",
    "mqtt_publish"
}

#### 3.2.2 Optimized Log Reading Function

Early filtering skips irrelevant logs, optimizing for efficiency in large-scale processing.

In [ ]:
def read_json_log(file_path, pcap_id, device_type):
    log_type = os.path.splitext(os.path.basename(file_path))[0]

    # Skip non–bruteforce logs early
    if log_type not in BRUTEFORCE_LOG_TYPES:
        return None

    try:
        df = pd.read_json(file_path, lines=True)
    except Exception as e:
        print(f"[!] Failed to read {file_path}: {e}")
        return None

    # Metadata
    df["pcap_id"] = pcap_id
    df["device_type"] = device_type
    df["log_type"] = log_type

    return df

#### 3.2.3 Collect Brute-Force Logs

Iterate folders, collecting only targeted logs to form the V2 dataset.

In [ ]:
bf_logs = []

for folder in pcap_folders:
    device_type = os.path.basename(folder)
    pcap_id = device_type  # or use another naming scheme

    for log_file in glob(os.path.join(folder, "*.log")):
        df = read_json_log(log_file, pcap_id, device_type)
        if df is not None and not df.empty:
            bf_logs.append(df)

len(bf_logs)

27

#### 3.2.4 Concatenate and Inspect V2 Dataset

Merge and check dimensions, ensuring focus on brute-force indicators.

In [ ]:
bf_df = pd.concat(bf_logs, ignore_index=True)

bf_df.shape

(34578, 53)

#### 3.2.5 Log Type Distribution

Analyze counts to confirm emphasis on relevant protocols (e.g., MQTT for IIoT messaging).

In [ ]:
bf_df["log_type"].value_counts()

log_type
mqtt_publish    28246
conn             6168
ssh               133
http               31
Name: count, dtype: int64

#### 3.2.6 Export V2 Dataset

Save for subsequent ML stages, such as feature extraction from authentication fields.

In [ ]:
output_csv = "merged_data_V2.csv"

bf_df.to_csv(output_csv, index=False)

import os
print("Saved to:", os.path.abspath(output_csv))

Saved to: C:\Users\User\Desktop\pc\projet ai\merged_data_V2.csv


## 4. Discussion and Next Steps

The generated datasets (V1 and V2) provide a foundation for advanced analysis. Future work includes:
- Feature engineering (e.g., aggregating attempts per time window).
- Model training using classifiers like Random Forests or Neural Networks.
- Evaluation metrics focused on precision/recall for imbalanced attack data.

This pipeline advances IIoT security by bridging raw traffic analysis with predictive modeling.